In [1]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & artiq_run " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, img_amp, freq_raman):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                # self.p.frequency_raman_transition = 147.2778e6 # .1 img amp
                self.p.frequency_raman_transition = {freq_raman}

                # self.xvar('t_continuous_rabi',np.linspace(0.,400.e-6,10))
                self.p.t_continuous_rabi = 300.e-6
                
                self.p.amp_imaging = {img_amp}
                
                # self.xvar('t_raman_pulse',np.linspace(0.,8.7e-6,7))

                self.p.hf_imaging_detuning = -568.e6 # 182.

                # self.xvar('hf_imaging_detuning_mid',np.arange(-680.,-610.,10)*1.e6)
                # self.xvar('hf_imaging_detuning_mid',[-670.e6,-640.e6])
                # self.p.hf_imaging_detuning_mid = -640.e6 # -635.e6
                self.p.hf_imaging_detuning_mid = -514.e6 # -1 --> 0
                
                # self.xvar('dimension_slm_mask',np.linspace(15.e-6,250.e-6,10))
                self.p.dimension_slm_mask = 20.e-6

                # self.p.phase_slm_mask = 0.371 * np.pi
                self.p.phase_slm_mask = 0.186 * np.pi

                # self.xvar('t_tweezer_hold',np.linspace(1.e-3,1.1e-3,10))
                self.p.t_tweezer_hold = 20.e-3
                self.p.t_tof = 20.e-6
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 10

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning_mid)
                # self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(squeeze=True)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)

                # self.raman.pulse(t=self.p.t_raman_pulse)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(3.e-6)
                self.raman.pulse(t=self.p.t_continuous_rabi)
                # delay(self.p.t_continuous_rabi)
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.imaging.set_power(1.,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [2]:
eBuilder = ExptBuilder()

In [3]:
img_amp = np.linspace(.3,1.9,20)
# np.random.shuffle(ts)
freq_raman = 119.4636e6 + np.linspace(81.205e3,599866.,20)
for f in range(20):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman = freq_raman[f]))
    eBuilder.run_expt()

0.3 119544805.0
0 Run id: 62905
 10 values of dummy. 10 total shots. 30 total images expected.
B:\_K\PotassiumData\2026-04-06\0062905_2026-04-06_16-40-27_hf_monitored_rabi.hdf5
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820

 Run ID: 62905

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.186 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.186, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle

In [13]:

relay = ethernet_relay

In [14]:
relay.source_off()

AttributeError: module 'waxx.control.ethernet_relay' has no attribute 'source_off'